In [0]:
%sql
DESCRIBE TABLE laddstolpar_df.bronze.trafa_t10026

In [0]:
%sql
SELECT regkom, ar,
       SUM(CASE WHEN drivmedel <> 't1' THEN CAST(itrfslut AS BIGINT) END) AS sum_fuels,
       MAX(CASE WHEN drivmedel =  't1' THEN CAST(itrfslut AS BIGINT) END) AS total_t1,
       MAX(CASE WHEN drivmedel = '105' THEN CAST(itrfslut AS BIGINT) END) AS phev
FROM laddstolpar_df.bronze.trafa_t10026
WHERE regkom IN ('0180', '2584') AND ar = '2025'
GROUP BY regkom, ar;

In [0]:
%sql DESCRIBE TABLE laddstolpar_df.bronze.elpris

In [0]:
%sql
SELECT COUNT(*)                               AS silver_rows,
       (SELECT COUNT(*) FROM laddstolpar_df.bronze.elpris) AS bronze_rows,
       MIN(time_start_utc), MAX(time_start_utc),
       SUM(CASE WHEN sek_per_kwh < 0 THEN 1 END) AS negative,
       MAX(length(split(CAST((SELECT MAX(SEK_per_kWh) FROM laddstolpar_df.bronze.elpris) AS STRING), '\\.')[1])) AS max_decimals_sample
FROM laddstolpar_df.silver.elpris;

In [0]:
%sql
SELECT elomrade, datum_lokal, time_start_utc, time_end_utc,
       unix_timestamp(time_end_utc) - unix_timestamp(time_start_utc) AS seconds,
       sek_per_kwh, _file
FROM laddstolpar_df.silver.elpris
WHERE unix_timestamp(time_end_utc) - unix_timestamp(time_start_utc) <> 900
   OR sek_per_kwh NOT BETWEEN -10 AND 50
ORDER BY time_start_utc, elomrade;

In [0]:
%sql
SELECT elomrade, time_start, time_end
FROM laddstolpar_df.bronze.elpris
WHERE elomrade = 'SE3'
  AND time_start BETWEEN '2025-10-26T02:30' AND '2025-10-26T03:15'
ORDER BY _file, time_start;

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(datum_lokal) AS with_date,
       MIN(datum_lokal), MAX(datum_lokal)
FROM laddstolpar_df.silver.elpris;

In [0]:
%sql
SELECT expected_rows, COUNT(*) AS zone_days,
       SUM(CASE WHEN n_rows = expected_rows THEN 1 ELSE 0 END) AS complete,
       SUM(CASE WHEN n_rows <> n_distinct_starts THEN 1 ELSE 0 END) AS with_duplicates
FROM laddstolpar_df.ops.elpris_day_check
GROUP BY expected_rows
ORDER BY expected_rows;